# 01 · Load & Clean LeBron's Per-Game Stats

**Before running:** get the data from [LeBron's Basketball-Reference page](https://www.basketball-reference.com/players/j/jamesle01.html):
1. Scroll to the **Per Game** table, then click **Share & Export > Get table as CSV (for Excel)**.
2. Copy the text that appears, paste it into a new file, and save it as `data/raw/lebron_per_game.csv`.

In [ ]:
import sqlite3
import pandas as pd

RAW = "../data/raw/lebron_per_game.csv"
PROCESSED = "../data/processed/lebron_per_game_clean.csv"
DB = "../data/processed/lebron.db"

## Step 1: First look

In [ ]:
df = pd.read_csv(RAW)
print(df.shape)
df.head()

In [ ]:
df.tail()  # the bottom usually has 'Career' or summary rows we don't want

## Step 2: Clean
Keep only real season rows (like `2012-13`), and make the column names SQL-friendly.

In [ ]:
df = df[df["Season"].astype(str).str.match(r"^\d{4}-\d{2}$")].copy()

# Basketball-Reference calls the team column 'Tm' or 'Team' depending on the page version
df = df.rename(columns={"Tm": "Team"})

# SQL-friendly names: lowercase, % -> _pct, 3P -> three_p, 2P -> two_p
df.columns = (df.columns.str.strip().str.lower()
              .str.replace("%", "_pct").str.replace("3p", "three_p").str.replace("2p", "two_p")
              .str.replace(r"[^a-z0-9_]", "_", regex=True))
df.head()

## Step 3: Add my own columns
This is where my story shows up in the data.

In [ ]:
df["age"] = df["age"].astype(int)
df["start_year"] = df["season"].str[:4].astype(int)

def era(year):
    if year <= 2009: return "1. Cleveland (1st stint)"
    if year <= 2013: return "2. Miami"
    if year <= 2017: return "3. Cleveland (2nd stint)"
    return "4. Lakers"

df["era"] = df["start_year"].apply(era)
df["watched_every_game"] = df["start_year"] >= 2012   # 6th grade, 2012-13
df[["season", "age", "team", "era", "watched_every_game", "pts"]]

## Step 4: Save to CSV (for Tableau) and SQLite (for DBeaver)

In [ ]:
df.to_csv(PROCESSED, index=False)

with sqlite3.connect(DB) as conn:
    df.to_sql("per_game", conn, if_exists="replace", index=False)

print("Saved", len(df), "seasons")

## Step 5: A quick first chart

In [ ]:
df.plot(x="age", y=["pts", "trb", "ast"], marker="o", figsize=(10, 5),
        title="LeBron: points, rebounds, assists by age");

## Notes: what surprised me?
*(write here, then copy the highlights into JOURNAL.md)*